In [ ]:
!pip install -q openai pandas numpy matplotlib

In [ ]:
import json
import time
import os
from getpass import getpass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openai import OpenAI

# Cole sua chave quando a célula pedir.
DEEPSEEK_API_KEY = getpass("Digite a chave:")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com"
)

print("Cliente DeepSeek configurado.")

Digite a chave:··········
Cliente DeepSeek configurado.


In [ ]:
# Carregamento do dataset Adult Income

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df = pd.read_csv(
    url,
    names=columns,
    skipinitialspace=True
)

print("Dataset carregado.")
print("Formato:", df.shape)
display(df.head())


Dataset carregado.
Formato: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
# Tools obrigatórias do trabalho

def listar_colunas():
    resultado = []
    for coluna in df.columns:
        resultado.append({
            "coluna": coluna,
            "tipo": str(df[coluna].dtype)
        })
    return resultado


def descrever_dados():
    return df.describe(include="all").fillna("").to_dict()


def contar_valores(coluna):
    if coluna not in df.columns:
        return {"erro": f"Coluna '{coluna}' não encontrada."}

    return df[coluna].value_counts().to_dict()


def _converter_valor_para_coluna(coluna, valor):
    """Converte valor recebido pelo LLM para o tipo da coluna, quando possível."""
    if coluna not in df.columns:
        return valor

    if pd.api.types.is_numeric_dtype(df[coluna]):
        try:
            return float(valor)
        except Exception:
            return valor

    return str(valor)


def filtrar(coluna, operador, valor):
    if coluna not in df.columns:
        return {"erro": f"Coluna '{coluna}' não encontrada."}

    valor_convertido = _converter_valor_para_coluna(coluna, valor)

    try:
        if operador == ">":
            filtrado = df[df[coluna] > valor_convertido]
        elif operador == "<":
            filtrado = df[df[coluna] < valor_convertido]
        elif operador == ">=":
            filtrado = df[df[coluna] >= valor_convertido]
        elif operador == "<=":
            filtrado = df[df[coluna] <= valor_convertido]
        elif operador == "==":
            filtrado = df[df[coluna] == valor_convertido]
        elif operador == "!=":
            filtrado = df[df[coluna] != valor_convertido]
        else:
            return {"erro": "Operador inválido. Use >, <, >=, <=, == ou !=."}

        return {
            "linhas_encontradas": int(len(filtrado)),
            "estatisticas": filtrado.describe(include="all").fillna("").to_dict()
        }

    except Exception as e:
        return {"erro": str(e)}


def agrupar_e_agregar(grupo, coluna, funcao):
    if grupo not in df.columns:
        return {"erro": f"Coluna de grupo '{grupo}' não encontrada."}

    if coluna not in df.columns:
        return {"erro": f"Coluna '{coluna}' não encontrada."}

    funcoes_permitidas = ["mean", "median", "sum", "min", "max", "count", "std"]

    if funcao not in funcoes_permitidas:
        return {"erro": f"Função inválida. Use uma destas: {funcoes_permitidas}"}

    try:
        resultado = df.groupby(grupo)[coluna].agg(funcao).to_dict()

        # Converte tipos numpy para tipos Python puros
        return {
            str(chave): float(valor) if isinstance(valor, (int, float, np.number)) else valor
            for chave, valor in resultado.items()
        }

    except Exception as e:
        return {"erro": str(e)}


def correlacao(coluna_a, coluna_b, metodo="pearson"):
    if coluna_a not in df.columns:
        return {"erro": f"Coluna '{coluna_a}' não encontrada."}

    if coluna_b not in df.columns:
        return {"erro": f"Coluna '{coluna_b}' não encontrada."}

    if not pd.api.types.is_numeric_dtype(df[coluna_a]):
        return {"erro": f"Coluna '{coluna_a}' não é numérica."}

    if not pd.api.types.is_numeric_dtype(df[coluna_b]):
        return {"erro": f"Coluna '{coluna_b}' não é numérica."}

    if metodo not in ["pearson", "spearman"]:
        return {"erro": "Método inválido. Use 'pearson' ou 'spearman'."}

    valor = df[coluna_a].corr(df[coluna_b], method=metodo)

    return {
        "coluna_a": coluna_a,
        "coluna_b": coluna_b,
        "metodo": metodo,
        "correlacao": float(valor)
    }


def detectar_outliers(coluna):
    if coluna not in df.columns:
        return {"erro": f"Coluna '{coluna}' não encontrada."}

    if not pd.api.types.is_numeric_dtype(df[coluna]):
        return {"erro": f"Coluna '{coluna}' não é numérica."}

    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = df[
        (df[coluna] < limite_inferior) |
        (df[coluna] > limite_superior)
    ]

    return {
        "coluna": coluna,
        "metodo": "IQR",
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "limite_inferior": float(limite_inferior),
        "limite_superior": float(limite_superior),
        "quantidade_outliers": int(len(outliers)),
        "percentual": round(len(outliers) / len(df) * 100, 2)
    }


def gerar_grafico(tipo, colunas):
    os.makedirs("graficos", exist_ok=True)

    if not colunas:
        return {"erro": "Informe pelo menos uma coluna."}

    for coluna in colunas:
        if coluna not in df.columns:
            return {"erro": f"Coluna '{coluna}' não encontrada."}

    caminho = f"graficos/grafico_{int(time.time())}.png"

    try:
        plt.figure(figsize=(8, 5))

        if tipo == "hist":
            df[colunas[0]].hist()
            plt.xlabel(colunas[0])
            plt.ylabel("Frequência")
            plt.title(f"Histograma de {colunas[0]}")

        elif tipo == "boxplot":
            df.boxplot(column=colunas[0])
            plt.title(f"Boxplot de {colunas[0]}")

        elif tipo == "scatter":
            if len(colunas) < 2:
                return {"erro": "Scatter exige duas colunas."}
            plt.scatter(df[colunas[0]], df[colunas[1]], alpha=0.5)
            plt.xlabel(colunas[0])
            plt.ylabel(colunas[1])
            plt.title(f"Dispersão: {colunas[0]} x {colunas[1]}")

        elif tipo == "barplot":
            df[colunas[0]].value_counts().head(10).plot(kind="bar")
            plt.xlabel(colunas[0])
            plt.ylabel("Contagem")
            plt.title(f"Top valores de {colunas[0]}")

        else:
            return {"erro": "Tipo inválido. Use hist, boxplot, scatter ou barplot."}

        plt.tight_layout()
        plt.savefig(caminho)
        plt.close()

        return {"arquivo": caminho}

    except Exception as e:
        return {"erro": str(e)}


MAPA_TOOLS = {
    "listar_colunas": listar_colunas,
    "descrever_dados": descrever_dados,
    "contar_valores": contar_valores,
    "filtrar": filtrar,
    "agrupar_e_agregar": agrupar_e_agregar,
    "correlacao": correlacao,
    "detectar_outliers": detectar_outliers,
    "gerar_grafico": gerar_grafico
}

print("Tools carregadas:", list(MAPA_TOOLS.keys()))


Tools carregadas: ['listar_colunas', 'descrever_dados', 'contar_valores', 'filtrar', 'agrupar_e_agregar', 'correlacao', 'detectar_outliers', 'gerar_grafico']


In [ ]:
# Esquema das tools para o DeepSeek

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "listar_colunas",
            "description": "Lista todas as colunas do dataset e seus tipos de dados.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "descrever_dados",
            "description": "Retorna estatísticas descritivas do dataset, incluindo colunas numéricas e categóricas.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "contar_valores",
            "description": "Conta a frequência dos valores de uma coluna categórica ou discreta.",
            "parameters": {
                "type": "object",
                "properties": {
                    "coluna": {
                        "type": "string",
                        "description": "Nome da coluna."
                    }
                },
                "required": ["coluna"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "filtrar",
            "description": "Filtra linhas do dataset a partir de uma condição simples e retorna estatísticas do subconjunto.",
            "parameters": {
                "type": "object",
                "properties": {
                    "coluna": {
                        "type": "string",
                        "description": "Nome da coluna usada no filtro."
                    },
                    "operador": {
                        "type": "string",
                        "enum": [">", "<", ">=", "<=", "==", "!="],
                        "description": "Operador do filtro."
                    },
                    "valor": {
                        "type": ["string", "number"],
                        "description": "Valor usado no filtro."
                    }
                },
                "required": ["coluna", "operador", "valor"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "agrupar_e_agregar",
            "description": "Agrupa o dataset por uma coluna e aplica uma agregação em outra coluna.",
            "parameters": {
                "type": "object",
                "properties": {
                    "grupo": {
                        "type": "string",
                        "description": "Coluna usada para agrupar."
                    },
                    "coluna": {
                        "type": "string",
                        "description": "Coluna que receberá a agregação."
                    },
                    "funcao": {
                        "type": "string",
                        "enum": ["mean", "median", "sum", "min", "max", "count", "std"],
                        "description": "Função de agregação."
                    }
                },
                "required": ["grupo", "coluna", "funcao"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "correlacao",
            "description": "Calcula correlação de Pearson ou Spearman entre duas colunas numéricas.",
            "parameters": {
                "type": "object",
                "properties": {
                    "coluna_a": {
                        "type": "string",
                        "description": "Primeira coluna numérica."
                    },
                    "coluna_b": {
                        "type": "string",
                        "description": "Segunda coluna numérica."
                    },
                    "metodo": {
                        "type": "string",
                        "enum": ["pearson", "spearman"],
                        "description": "Método de correlação. Padrão: pearson."
                    }
                },
                "required": ["coluna_a", "coluna_b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "detectar_outliers",
            "description": "Detecta outliers em uma coluna numérica usando o método IQR.",
            "parameters": {
                "type": "object",
                "properties": {
                    "coluna": {
                        "type": "string",
                        "description": "Coluna numérica para análise de outliers."
                    }
                },
                "required": ["coluna"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gerar_grafico",
            "description": "Gera um gráfico e salva como imagem. Tipos: hist, boxplot, scatter, barplot.",
            "parameters": {
                "type": "object",
                "properties": {
                    "tipo": {
                        "type": "string",
                        "enum": ["hist", "boxplot", "scatter", "barplot"],
                        "description": "Tipo de gráfico."
                    },
                    "colunas": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Lista de colunas usadas no gráfico."
                    }
                },
                "required": ["tipo", "colunas"]
            }
        }
    }
]

print("Schemas declarados:", len(tools_schema))


Schemas declarados: 8


In [ ]:
# Executor genérico e loop do agente

def _json_seguro(obj):
    return json.dumps(obj, ensure_ascii=False, default=str)


def _conteudo_tool_limitado(resultado, limite=12000):
    texto = _json_seguro(resultado)
    if len(texto) > limite:
        return texto[:limite] + "\n... [resultado truncado para caber no contexto]"
    return texto


def executar_tool(nome_tool, argumentos):
    if nome_tool not in MAPA_TOOLS:
        return {"erro": f"Tool '{nome_tool}' não existe."}

    try:
        funcao = MAPA_TOOLS[nome_tool]
        return funcao(**argumentos)
    except Exception as e:
        return {"erro": str(e)}


def executar_agente(pergunta, max_iteracoes=5, verbose=True):
    inicio = time.time()
    logs = []

    contexto_colunas = listar_colunas()

    messages = [
        {
            "role": "system",
            "content": f"""
Você é um agente de análise exploratória de dados em português.

Você analisa o dataset Adult Income usando tools Python/pandas.
Nunca invente números, colunas ou resultados.
Sempre que a pergunta exigir dados do CSV, chame uma tool.
Se a pergunta for ambígua, inválida ou impossível com as colunas disponíveis, explique o problema.
Depois de receber o resultado da tool, responda em português claro.

Colunas disponíveis:
{contexto_colunas}
"""
        },
        {
            "role": "user",
            "content": pergunta
        }
    ]

    for iteracao in range(max_iteracoes):
        resposta = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        mensagem = resposta.choices[0].message

        if not mensagem.tool_calls:
            latencia = time.time() - inicio

            return {
                "pergunta": pergunta,
                "resposta": mensagem.content,
                "logs": logs,
                "latencia_segundos": round(latencia, 3),
                "tool_calls": len(logs)
            }

        # Adiciona a mensagem do assistente com tool_calls ao histórico
        messages.append({
            "role": "assistant",
            "content": mensagem.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in mensagem.tool_calls
            ]
        })

        for tool_call in mensagem.tool_calls:
            nome_tool = tool_call.function.name

            try:
                argumentos = json.loads(tool_call.function.arguments or "{}")
            except Exception:
                argumentos = {}

            resultado = executar_tool(nome_tool, argumentos)

            log = {
                "iteracao": iteracao + 1,
                "tool": nome_tool,
                "argumentos": argumentos,
                "resultado": resultado
            }
            logs.append(log)

            if verbose:
                print(f"Tool chamada: {nome_tool}")
                print(f"Argumentos: {argumentos}")
                print(f"Resultado: {str(resultado)[:700]}")
                print("-" * 80)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": nome_tool,
                "content": _conteudo_tool_limitado(resultado)
            })

    latencia = time.time() - inicio

    return {
        "pergunta": pergunta,
        "resposta": "O agente atingiu o limite de iterações antes de concluir.",
        "logs": logs,
        "latencia_segundos": round(latencia, 3),
        "tool_calls": len(logs)
    }


print("Agente pronto.")


Agente pronto.


In [ ]:
# Testes principais do agente

perguntas_teste = [
    "Quais colunas existem no dataset?",
    "Qual é a distribuição da coluna income?",
    "Existe correlação entre idade e horas trabalhadas por semana?",
    "Qual é a média de idade por sexo?",
    "Existem outliers na coluna hours_per_week?",
    "Gere um histograma da coluna age.",
    "Qual é a melhor coluna?"
]

for pergunta in perguntas_teste:
    print("\nPERGUNTA:", pergunta)
    saida = executar_agente(pergunta, verbose=True)
    print("RESPOSTA FINAL:")
    print(saida["resposta"])
    print("Tool calls:", saida["tool_calls"], "| Latência:", saida["latencia_segundos"])
    print("=" * 100)



PERGUNTA: Quais colunas existem no dataset?
Tool chamada: listar_colunas
Argumentos: {}
Resultado: [{'coluna': 'age', 'tipo': 'int64'}, {'coluna': 'workclass', 'tipo': 'object'}, {'coluna': 'fnlwgt', 'tipo': 'int64'}, {'coluna': 'education', 'tipo': 'object'}, {'coluna': 'education_num', 'tipo': 'int64'}, {'coluna': 'marital_status', 'tipo': 'object'}, {'coluna': 'occupation', 'tipo': 'object'}, {'coluna': 'relationship', 'tipo': 'object'}, {'coluna': 'race', 'tipo': 'object'}, {'coluna': 'sex', 'tipo': 'object'}, {'coluna': 'capital_gain', 'tipo': 'int64'}, {'coluna': 'capital_loss', 'tipo': 'int64'}, {'coluna': 'hours_per_week', 'tipo': 'int64'}, {'coluna': 'native_country', 'tipo': 'object'}, {'coluna': 'income', 'tipo': 'object'}]
--------------------------------------------------------------------------------
RESPOSTA FINAL:
O dataset **Adult Income** possui **15 colunas** no total. Aqui estão elas organizadas por tipo:

### 🔢 Colunas Numéricas (int64)
| Coluna | Descrição |
|---

In [ ]:
# Salvar logs de uma pergunta em arquivo JSON

saida = executar_agente(
    "Existe correlação entre idade e horas trabalhadas por semana?",
    verbose=True
)

os.makedirs("logs", exist_ok=True)

with open("logs/exemplo_execucao.json", "w", encoding="utf-8") as f:
    json.dump(saida, f, ensure_ascii=False, indent=2, default=str)

print("Log salvo em logs/exemplo_execucao.json")


Tool chamada: correlacao
Argumentos: {'coluna_a': 'age', 'coluna_b': 'hours_per_week', 'metodo': 'pearson'}
Resultado: {'coluna_a': 'age', 'coluna_b': 'hours_per_week', 'metodo': 'pearson', 'correlacao': 0.06875570750955737}
--------------------------------------------------------------------------------
Log salvo em logs/exemplo_execucao.json


In [ ]:
# Dataset de Avaliação para o Benchmark

dataset_avaliacao = [

    # Perguntas factuais
    {
        "id": 1,
        "pergunta": "Quais são as colunas presentes na tabela?",
        "categoria": "Factual",
        "tool_esperada": "listar_colunas",
        "tipo_resposta": "lista_strings",
        "gabarito": ["age", "workclass", "education", "marital-status", "occupation", "race", "sex", "hours-per-week", "native-country", "income"]
    },
    {
        "id": 2,
        "pergunta": "Quantas linhas e quantas colunas existem no dataset?",
        "categoria": "Factual",
        "tool_esperada": "ver_dimensoes",
        "tipo_resposta": "lista_strings",
        "gabarito": ["32561", "15"]
    },
    {
        "id": 3,
        "pergunta": "Quais são os tipos de dados de cada coluna?",
        "categoria": "Factual",
        "tool_esperada": "listar_colunas",
        "tipo_resposta": "lista_strings",
        "gabarito": ["int64", "object"]
    },
    {
        "id": 4,
        "pergunta": "Quantos valores nulos ou faltantes existem na coluna occupation?",
        "categoria": "Factual",
        "tool_esperada": "contar_nulos",
        "tipo_resposta": "numero_inteiro",
        "gabarito": 0
    },
    {
        "id": 5,
        "pergunta": "Quantas pessoas existem no dataset que pertencem ao sexo feminino?",
        "categoria": "Factual",
        "tool_esperada": "contar_valores",
        "tipo_resposta": "numero_inteiro",
        "gabarito": 10771
    },
    {
        "id": 6,
        "pergunta": "Qual é o valor máximo encontrado na coluna age?",
        "categoria": "Factual",
        "tool_esperada": "resumo_estatistico",
        "tipo_resposta": "numero_inteiro",
        "gabarito": 90
    },
    {
        "id": 7,
        "pergunta": "Qual é a média de idade (age) cadastrada no sistema?",
        "categoria": "Factual",
        "tool_esperada": "resumo_estatistico",
        "tipo_resposta": "numero_float",
        "gabarito": 38.58
    },
    {
        "id": 8,
        "pergunta": "Exiba as primeiras 5 linhas do conjunto de dados.",
        "categoria": "Factual",
        "tool_esperada": "visualizar_amostra",
        "tipo_resposta": "categorica",
        "gabarito": "age"
    },
    {
        "id": 9,
        "pergunta": "Quais são os diferentes países de origem (native-country) únicos cadastrados?",
        "categoria": "Factual",
        "tool_esperada": "listar_valores_unicos",
        "tipo_resposta": "lista_strings",
        "gabarito": ["United-States", "Mexico", "Philippines", "Germany", "Canada"]
    },
    {
        "id": 10,
        "pergunta": "A coluna income possui valores duplicados?",
        "categoria": "Factual",
        "tool_esperada": "verificar_duplicados",
        "tipo_resposta": "categorica",
        "gabarito": "sim"  # Variável binária, com certeza possui muitos valores duplicados
    },

    # Peguntas analíticas
    {
        "id": 11,
        "pergunta": "Qual é a média de horas trabalhadas por semana agrupada por classe de renda (income)?",
        "categoria": "Analítica",
        "tool_esperada": "agrupar_e_agregar",
        "tipo_resposta": "dict_numerico",
        "gabarito": {"<=50k": 38.84, ">50k": 45.47}
    },
    {
        "id": 12,
        "pergunta": "Existe alguma correlação linear entre a idade e as horas semanais trabalhadas?",
        "categoria": "Analítica",
        "tool_esperada": "correlacao",
        "tipo_resposta": "numero_float",
        "gabarito": 0.068  # Valor da correlação de Pearson entre age e hours-per-week
    },
    {
        "id": 13,
        "pergunta": "Gere um gráfico de dispersão cruzando age e hours_per_week.",
        "categoria": "Analítica",
        "tool_esperada": "gerar_grafico",
        "tipo_resposta": "categorica",
        "gabarito": "gráfico"
    },
    {
        "id": 14,
        "pergunta": "Qual é a distribuição percentual do nível de escolaridade (education) no dataset?",
        "categoria": "Analítica",
        "tool_esperada": "contar_valores",
        "tipo_resposta": "lista_strings",
        "gabarito": ["HS-grad", "Some-college", "Bachelors", "Masters"]
    },
    {
        "id": 15,
        "pergunta": "Mostre um histograma mostrando a distribuição da idade da população amostrada.",
        "categoria": "Analítica",
        "tool_esperada": "gerar_grafico",
        "tipo_resposta": "categorica",
        "gabarito": "histograma"
    },
    {
        "id": 16,
        "pergunta": "Existem outliers ou valores discrepantes gritantes na coluna de horas trabalhadas?",
        "categoria": "Analítica",
        "tool_esperada": "detectar_outliers",
        "tipo_resposta": "categorica",
        "gabarito": "outlier"
    },
    {
        "id": 17,
        "pergunta": "Qual é a proporção de homens e mulheres que ganham mais de 50k por ano?",
        "categoria": "Analítica",
        "tool_esperada": "agrupar_e_agregar",
        "tipo_resposta": "dict_numerico",
        "gabarito": {"Male": 0.30, "Female": 0.10}  # Proporções aproximadas internas de cada género
    },
    {
        "id": 18,
        "pergunta": "Crie um gráfico de barras vertical comparando a quantidade de pessoas por raça (race).",
        "categoria": "Analítica",
        "tool_esperada": "gerar_grafico",
        "tipo_resposta": "categorica",
        "gabarito": "barra"
    },
    {
        "id": 19,
        "pergunta": "Qual profissão (occupation) apresenta a maior mediana de horas trabalhadas por semana?",
        "categoria": "Analítica",
        "tool_esperada": "agrupar_e_agregar",
        "tipo_resposta": "categorica",
        "gabarito": "Farming-fishing"  # Ou Exec-managerial dependendo de variações de tratamento
    },
    {
        "id": 20,
        "pergunta": "Se filtrarmos apenas pessoas maiores de 40 anos, qual é o nível de instrução mais comum?",
        "categoria": "Analítica",
        "tool_esperada": "filtrar_e_analisar",
        "tipo_resposta": "categorica",
        "gabarito": "HS-grad"
    },
    {
        "id": 21,
        "pergunta": "A variância da idade é muito alta para as pessoas que trabalham no setor privado?",
        "categoria": "Analítica",
        "tool_esperada": "agrupar_e_agregar",
        "tipo_resposta": "numero_float",
        "gabarito": 136.1  # Variância aproximada (Desvio padrão ~11.66 elevado ao quadrado)
    },
    {
        "id": 22,
        "pergunta": "Apresente uma matriz de correlação entre todas as variáveis numéricas disponíveis.",
        "categoria": "Analítica",
        "tool_esperada": "correlacao",
        "tipo_resposta": "lista_strings",
        "gabarito": ["age", "education-num", "capital-gain", "hours-per-week"]
    },
    {
        "id": 23,
        "pergunta": "Qual o desvio padrão da coluna capital-gain?",
        "categoria": "Analítica",
        "tool_esperada": "resumo_estatistico",
        "tipo_resposta": "numero_float",
        "gabarito": 7385.29
    },
    {
        "id": 24,
        "pergunta": "Gere um boxplot da idade separado por gênero para identificar assimetrias.",
        "categoria": "Analítica",
        "tool_esperada": "gerar_grafico",
        "tipo_resposta": "categorica",
        "gabarito": "boxplot"
    },
    {
        "id": 25,
        "pergunta": "Quais são as 3 ocupações menos frequentes no dataset?",
        "categoria": "Analítica",
        "tool_esperada": "contar_valores",
        "tipo_resposta": "lista_strings",
        "gabarito": ["Armed-Forces", "Priv-house-serv", "Protective-serv"]
    },

    # Perguntas ambíguas / inválidas

    {
        "id": 26,
        "pergunta": "Faz um negócio aí com a coluna age.",
        "categoria": "Ambígua/Inválida",
        "tool_esperada": "nenhuma",
        "tipo_resposta": "categorica",
        "gabarito": "especifique"  # Espera uma resposta de clarificação do LLM
    },
    {
        "id": 27,
        "pergunta": "Qual é a temperatura média em Curitiba hoje?",
        "categoria": "Ambígua/Inválida",
        "tool_esperada": "nenhuma",
        "tipo_resposta": "categorica",
        "gabarito": "escopo"  # Deve indicar que a pergunta está fora do escopo do arquivo
    },
    {
        "id": 28,
        "pergunta": "Me diga quem é o usuário mais rico cruzando com o banco de dados do Google.",
        "categoria": "Ambígua/Inválida",
        "tool_esperada": "nenhuma",
        "tipo_resposta": "categorica",
        "gabarito": "não consigo"  # Deve recusar por falta de acesso externo/privacidade
    },
    {
        "id": 29,
        "pergunta": "Quero que você exclua todas as linhas onde a idade seja menor que 18.",
        "categoria": "Ambígua/Inválida",
        "tool_esperada": "nenhuma",
        "tipo_resposta": "categorica",
        "gabarito": "apenas leitura"  # Deve recusar mutações de dados
    },
    {
        "id": 30,
        "pergunta": "O que você acha da política econômica atual?",
        "categoria": "Ambígua/Inválida",
        "tool_esperada": "nenhuma",
        "tipo_resposta": "categorica",
        "gabarito": "opinião"  # Resposta neutra indicando incapacidade de emitir juízos de valor
    }
]

In [ ]:
# Avaliação além da tool correta
import re

def avaliar_resposta_pragmatica(resposta_llm, tipo_resposta, gabarito, tolerancia=0.05):
    if not resposta_llm:
        return 0

    # Normaliza o texto: remove hifens e underlines para evitar erros de grafia das colunas
    resposta_limpa = str(resposta_llm).lower().replace("-", " ").replace("_", " ")

    # Helper para normalizar itens de listas/gabaritos
    def norm(v): return str(v).lower().replace("-", " ").replace("_", " ")

    # NÚMERO INTEIRO / FLOAT
    if tipo_resposta in ["numero_inteiro", "numero_float"]:
        # Remove pontos finais de fim de frase grudados em números
        numeros_encontrados = re.findall(r"[-+]?\d*\.\d+|\d+", str(resposta_llm))
        if not numeros_encontrados:
            return 0

        primeiro_numero = float(numeros_encontrados[0])
        gabarito_num = float(gabarito)

        if gabarito_num == 0:
            return 1 if primeiro_numero == 0 else 0
        return 1 if abs(primeiro_numero - gabarito_num) / gabarito_num <= tolerancia else 0

    # LISTA DE STRINGS
    elif tipo_resposta == "lista_strings":
        for item in gabarito:
            if norm(item) not in resposta_limpa:
                return 0
        return 1

    # DICIONÁRIO NUMÉRICO
    elif tipo_resposta == "dict_numerico":
        for chave, valor_esperado in gabarito.items():
            chave_limpa = norm(chave)
            if chave_limpa not in resposta_limpa:
                return 0

            # Pega o trecho após a chave para buscar o número correspondente
            trecho = resposta_limpa.split(chave_limpa)[1][:50]
            numeros_trecho = re.findall(r"[-+]?\d*\.\d+|\d+", trecho)
            if not numeros_trecho:
                return 0

            valor_encontrado = float(numeros_trecho[0])
            if valor_esperado == 0:
                if valor_encontrado != 0: return 0
            elif abs(valor_encontrado - valor_esperado) / valor_esperado > tolerancia:
                return 0
        return 1

    # CATEGÓRICA / PALAVRA-CHAVE
    elif tipo_resposta == "categorica":
        # Se for ambígua/inválida e o modelo recusou/pediu ajuda, consideramos correto
        if norm(gabarito) in ["especifique", "escopo", "não consigo", "apenas leitura", "opinião"]:
            palavras_recusa = ["escopo", "não posso", "especifique", "desculpe", "apenas leitura", "não consigo", "inválida", "por favor"]
            if any(p in resposta_limpa for p in palavras_recusa):
                return 1
        return 1 if norm(gabarito) in resposta_limpa else 0

    return 0

In [ ]:
import pandas as pd

def rodar_benchmark_completo(dataset):
    resultados = []

    print(f"Iniciando avaliação de {len(dataset)} cenários do dataset...\n")
    print("=" * 80)

    for item in dataset:
        print(f"Testando ID {item['id']} [{item['categoria']}]")
        print(f"Pergunta: '{item['pergunta']}'")

        # Executa o agente
        saida_agente = executar_agente(item['pergunta'], verbose=False)
        resposta_texto = saida_agente['resposta']

        # Mapeamento de logs
        tool_utilizada = saida_agente['logs'][0]['tool'] if saida_agente['logs'] else "nenhuma"
        acertou_tool = 1 if tool_utilizada == item['tool_esperada'] else 0

        acertou_conteudo = avaliar_resposta_pragmatica(
            resposta_texto,
            item['tipo_resposta'],
            item['gabarito']
        )

        resultados.append({
            "id": item['id'],
            "pergunta": item['pergunta'],
            "categoria": item['categoria'],
            "tool_esperada": item['tool_esperada'],
            "tool_utilizada": tool_utilizada,
            "acertou_tool": acertou_tool,
            "tipo_resposta": item['tipo_resposta'],
            "acertou_conteudo": acertou_conteudo,
            "latencia": saida_agente['latencia_segundos'],
            "tool_calls": saida_agente['tool_calls'],
            "resposta_agente": resposta_texto
        })

        print(f"Tool Esperada: {item['tool_esperada']} | Utilizada: {tool_utilizada}")
        print(f"Acertou Tool? {'✅ SIM' if acertou_tool == 1 else '❌ NÃO'}")
        print(f" Acertou Conteúdo? {'✅ SIM' if acertou_conteudo == 1 else '❌ NÃO'}")
        print("-" * 80)

    return pd.DataFrame(resultados)

# Execução
df_res = rodar_benchmark_completo(dataset_avaliacao)

Iniciando avaliação de 30 cenários do dataset...

Testando ID 1 [Factual]
Pergunta: 'Quais são as colunas presentes na tabela?'
Tool Esperada: listar_colunas | Utilizada: listar_colunas
Acertou Tool? ✅ SIM
 Acertou Conteúdo? ✅ SIM
--------------------------------------------------------------------------------
Testando ID 2 [Factual]
Pergunta: 'Quantas linhas e quantas colunas existem no dataset?'
Tool Esperada: ver_dimensoes | Utilizada: descrever_dados
Acertou Tool? ❌ NÃO
 Acertou Conteúdo? ❌ NÃO
--------------------------------------------------------------------------------
Testando ID 3 [Factual]
Pergunta: 'Quais são os tipos de dados de cada coluna?'
Tool Esperada: listar_colunas | Utilizada: listar_colunas
Acertou Tool? ✅ SIM
 Acertou Conteúdo? ✅ SIM
--------------------------------------------------------------------------------
Testando ID 4 [Factual]
Pergunta: 'Quantos valores nulos ou faltantes existem na coluna occupation?'
Tool Esperada: contar_nulos | Utilizada: contar_va

In [ ]:
# import os
# import json
# import time

# # 1. Estatísticas Gerais
# acuracia_tool_geral = df_res['acertou_tool'].mean() * 100
# acuracia_texto_geral = df_res['acertou_conteudo'].mean() * 100
# latencia_media = df_res['latencia'].mean()
# total_chamadas = df_res['tool_calls'].sum()

# print("==================================================================")
# print("                  RESULTADOS GERAIS DO BENCHMARK                  ")
# print("==================================================================")
# print(f"Acurácia de Seleção de Tool (Intenção):  {acuracia_tool_geral:.2f}%")
# print(f"Acurácia Pragmática de Conteúdo (Texto): {acuracia_texto_geral:.2f}%")
# print(f"Tempo Médio de Resposta:                 {latencia_media:.2f}s")
# print("==================================================================\n")

# # 2. Exibição da Tabela por Categoria
# df_categorias = df_res.groupby('categoria').agg(
#     total_testes=('id', 'count'),
#     taxa_acerto_tool=('acertou_tool', 'mean'),
#     taxa_acerto_conteudo=('acertou_conteudo', 'mean'),
#     tempo_medio=('latencia', 'mean')
# ).reset_index()

# df_categorias['taxa_acerto_tool'] = (df_categorias['taxa_acerto_tool'] * 100).map('{:.2f}%'.format)
# df_categorias['taxa_acerto_conteudo'] = (df_categorias['taxa_acerto_conteudo'] * 100).map('{:.2f}%'.format)
# df_categorias['tempo_medio'] = df_categorias['tempo_medio'].map('{:.2f}s'.format)
# display(df_categorias)

# # 3. Exportação para a pasta de avaliação
# os.makedirs("results_evaluation", exist_ok=True)
# df_res.to_csv("results_evaluation/benchmark_detalhado.csv", index=False, encoding="utf-8")

# report_final = {
#     "modelo_testado": "deepseek-chat",
#     "total_testes": len(df_res),
#     "acuracia_selecao_tool": f"{acuracia_tool_geral:.2f}%",
#     "acuracia_conteudo_textual": f"{acuracia_texto_geral:.2f}%",
#     "latencia_media_segundos": round(latencia_media, 2),
#     "timestamp_execucao": time.strftime("%Y-%m-%d %H:%M:%S")
# }

# with open("results_evaluation/summary_metrics.json", "w", encoding="utf-8") as f:
#     json.dump(report_final, f, ensure_ascii=False, indent=4)

In [ ]:
import os
import json
import time

# 1. Estatísticas Gerais
acuracia_tool_geral = df_res['acertou_tool'].mean() * 100
acuracia_texto_geral = df_res['acertou_conteudo'].mean() * 100
latencia_media = df_res['latencia'].mean()
total_chamadas = df_res['tool_calls'].sum()
media_tool_calls = df_res['tool_calls'].mean()

# Taxa de sucesso: execução sem erro/crash
if 'erro' in df_res.columns:
    taxa_sucesso = df_res['erro'].isna().mean() * 100
else:
    taxa_sucesso = df_res['resposta_agente'].notna().mean() * 100

# Custo médio estimado
# Ajuste esses valores se vocês tiverem o custo real do DeepSeek usado.
# Exemplo conservador: se não coletaram tokens, deixem como estimativa simbólica.
custo_total_usd = 0.00
custo_medio_usd = custo_total_usd / len(df_res)

print("==================================================================")
print("                  RESULTADOS GERAIS DO BENCHMARK                  ")
print("==================================================================")
print(f"Acurácia de Seleção de Tool (Intenção):  {acuracia_tool_geral:.2f}%")
print(f"Acurácia Pragmática de Conteúdo (Texto): {acuracia_texto_geral:.2f}%")
print(f"Taxa de Execução Bem-Sucedida:           {taxa_sucesso:.2f}%")
print(f"Nº Médio de Tool Calls por Pergunta:     {media_tool_calls:.2f}")
print(f"Tempo Médio de Resposta / Latência:      {latencia_media:.2f}s")
#print(f"Custo Médio por Pergunta:                US$ {custo_medio_usd:.6f}")
print("==================================================================\n")

# 2. Exibição da Tabela por Categoria
df_categorias = df_res.groupby('categoria').agg(
    total_testes=('id', 'count'),
    taxa_acerto_tool=('acertou_tool', 'mean'),
    taxa_acerto_conteudo=('acertou_conteudo', 'mean'),
    taxa_sucesso=('resposta_agente', lambda x: x.notna().mean()),
    media_tool_calls=('tool_calls', 'mean'),
    tempo_medio=('latencia', 'mean')
).reset_index()

df_categorias['taxa_acerto_tool'] = (df_categorias['taxa_acerto_tool'] * 100).map('{:.2f}%'.format)
df_categorias['taxa_acerto_conteudo'] = (df_categorias['taxa_acerto_conteudo'] * 100).map('{:.2f}%'.format)
df_categorias['taxa_sucesso'] = (df_categorias['taxa_sucesso'] * 100).map('{:.2f}%'.format)
df_categorias['media_tool_calls'] = df_categorias['media_tool_calls'].map('{:.2f}'.format)
df_categorias['tempo_medio'] = df_categorias['tempo_medio'].map('{:.2f}s'.format)

display(df_categorias)

# 3. Exportação para a pasta de avaliação
os.makedirs("results_evaluation", exist_ok=True)
df_res.to_csv("results_evaluation/benchmark_detalhado.csv", index=False, encoding="utf-8")

report_final = {
    "modelo_testado": "deepseek-chat",
    "total_testes": int(len(df_res)),
    "acuracia_selecao_tool": f"{acuracia_tool_geral:.2f}%",
    "acuracia_conteudo_textual": f"{acuracia_texto_geral:.2f}%",
    "taxa_execucao_bem_sucedida": f"{taxa_sucesso:.2f}%",
    "media_tool_calls_por_pergunta": round(media_tool_calls, 2),
    "latencia_media_segundos": round(latencia_media, 2),
    "custo_total_estimado_usd": round(custo_total_usd, 6),
   # "custo_medio_por_pergunta_usd": round(custo_medio_usd, 6),
    "timestamp_execucao": time.strftime("%Y-%m-%d %H:%M:%S")
}

with open("results_evaluation/summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(report_final, f, ensure_ascii=False, indent=4)

print("Arquivos salvos:")
print("- results_evaluation/benchmark_detalhado.csv")
print("- results_evaluation/summary_metrics.json")

                  RESULTADOS GERAIS DO BENCHMARK                  
Acurácia de Seleção de Tool (Intenção):  43.33%
Acurácia Pragmática de Conteúdo (Texto): 63.33%
Taxa de Execução Bem-Sucedida:           100.00%
Nº Médio de Tool Calls por Pergunta:     2.00
Tempo Médio de Resposta / Latência:      5.22s



,categoria,total_testes,taxa_acerto_tool,taxa_acerto_conteudo,taxa_sucesso,media_tool_calls,tempo_medio
0,Ambígua/Inválida,5,40.00%,40.00%,100.00%,2.00,5.96s
1,Analítica,15,60.00%,66.67%,100.00%,2.60,6.04s
2,Factual,10,20.00%,70.00%,100.00%,1.10,3.62s


Arquivos salvos:
- results_evaluation/benchmark_detalhado.csv
- results_evaluation/summary_metrics.json


In [ ]:
with open("benchmark.json", "w", encoding="utf-8") as f: #benchmark.json
    json.dump(dataset_avaliacao, f, ensure_ascii=False, indent=2)

In [ ]:
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 11.3 MB/s eta 0:00:00


In [ ]:
# Interfaxe com gradio

import gradio as gr
import re
import os
import pandas as pd

# PREPARAÇÃO DOS DADOS DA ABA 2
try:

    df_amostra = df.head(10)

    # Resumo Numérico (Apenas colunas de número: age, hours_per_week, etc.)
    df_numerico = df.describe(include=[object]).reset_index() if df.select_dtypes(include=['number']).empty else df.describe(include=['number']).round(2).reset_index()
    df_numerico.rename(columns={'index': 'Métrica Numérica'}, inplace=True)

    # Resumo Categórico (Apenas colunas de texto: workclass, occupation, etc.)
    df_categorico = df.describe(include=['object', 'category']).reset_index()
    df_categorico.rename(columns={'index': 'Métrica Categórica'}, inplace=True)

except NameError:
    # Fallback caso o df não esteja na memória da célula
    df_amostra = pd.DataFrame({"Aviso": ["Carregue o dataset antes de rodar o Gradio."]})
    df_numerico = pd.DataFrame({"Aviso": ["Sem dados numéricos."]})
    df_categorico = pd.DataFrame({"Aviso": ["Sem dados categóricos."]})

# Função do Agente
def interface_agente_com_grafico(pergunta):
    if not pergunta.strip():
        return "Por favor, digite uma pergunta válida.", None, "0.00 segundos"

    saida = executar_agente(pergunta, verbose=False)

    resposta_texto = saida.get('resposta', 'Sem resposta.')
    latencia = saida.get('latencia_segundos', 0.0)

    # Remove os links de imagem do Markdown (![texto](caminho)) para não poluir a tela
    resposta_limpa = re.sub(r'!\[.*?\]\(.*?\)', '', resposta_texto).strip()

    caminho_imagem = None
    if saida.get('logs'):
        for log in saida['logs']:
            if log.get('tool') == 'gerar_grafico' and isinstance(log.get('resultado'), dict):
                possivel_caminho = log['resultado'].get('arquivo')
                if possivel_caminho and os.path.exists(possivel_caminho):
                    caminho_imagem = possivel_caminho
                    break

    return resposta_limpa, caminho_imagem, f"{latencia:.2f} segundos"

# Construção da Interface
with gr.Blocks(title="🤖 Agente EDA", theme=gr.themes.Soft()) as app:
    gr.Markdown("# Plataforma Inteligente de EDA")

    # ABA 1: O AGENTE
    with gr.Tab("🤖 Conversar com o Agente"):
        gr.Markdown("Pergunte qualquer coisa sobre as variáveis ou peça plots personalizados.")
        with gr.Row():
            with gr.Column(scale=2):
                input_texto = gr.Textbox(
                    label="Sua Pergunta",
                    placeholder="Ex: Qual é a média de idade por gênero?",
                    lines=3
                )
                botao_enviar = gr.Button("🚀 Executar Análise", variant="primary")
                output_tempo = gr.Label(label="⏱️ Tempo de Processamento")

            with gr.Column(scale=3):
                output_resposta = gr.Markdown(value="*A resposta do agente aparecerá aqui...*")
                output_grafico = gr.Image(label="📊 Gráfico Gerado (se houver)", type="filepath")

        botao_enviar.click(
            fn=interface_agente_com_grafico,
            inputs=input_texto,
            outputs=[output_resposta, output_grafico, output_tempo]
        )

    # ABA 2: VISUALIZAÇÃO DOS DADOS
    with gr.Tab("📊 Visualização do Dataset"):
        gr.Markdown("### 📄 Amostra Dataset Adult Income (10 primeiros valores)")
        # Exibe os dados de forma interativa (permite scroll, ordenação, etc.)
        gr.DataFrame(df_amostra, interactive=False)

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🔢 Estatísticas de Variáveis Numéricas")
                gr.Markdown("*Analisa médias, mínimos, máximos e quartis.*")
                gr.DataFrame(df_numerico, interactive=False)

            with gr.Column():
                gr.Markdown("### 🔤 Estatísticas de Variáveis Categóricas")
                gr.Markdown("*Analisa valores únicos (unique), o termo mais frequente (top) e a sua frequência (freq).*")
                gr.DataFrame(df_categorico, interactive=False)

# Iniciar a aplicação
app.launch(inline=True, share=True)

/tmp/ipykernel_1025/1272733124.py:52: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🤖 Agente EDA", theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://644b2fad287d776bb6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
